In [ ]:
import json
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report

from decision_maker import TCAAnalyzer

# 1. Load dataset
file_path = "../data/conversations.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

def extract_user_messages(sample):
    messages = sample.get("messages", [])
    user_msgs = []
    for m_str in messages:
        try:
            m = json.loads(m_str) if isinstance(m_str, str) else m_str
            if m.get("role") == "user":
                user_msgs.append(m.get("body", ""))
        except:
            continue
    return user_msgs

# 3. Initialize tracking lists
tca = TCAAnalyzer()
y_true = []
y_pred = []

# 4. Run loop
for idx, sample in enumerate(data): # Increased sample size for better accuracy stats
    tca.reset() 
    user_messages = extract_user_messages(sample)

    actual_label = 1
    y_true.append(actual_label)
    
    turn_results = []
    is_blocked = False
    
    for turn_idx, msg in enumerate(user_messages):
        result = tca.process_turn(msg)
        decision = result.progression_summary.summary
        flags = result.progression_summary.dominant_flag
        
        
        if "BLOCK" in decision or flags in ["SPIKE", "OBFUSCATION"]:
            is_blocked = True
            turn_results.append(1)
            break 
        else:
            turn_results.append(0)
    
    # Record Prediction
    y_pred.append(1 if is_blocked else 0)
  

# 5. Final Accuracy Calculation
print("\n" + "="*30)
print("FINAL EVALUATION")
print("="*30)
acc = accuracy_score(y_true, y_pred)
print(f"Overall Accuracy: {acc * 100:.2f}%")

# Detailed Report (Precision, Recall, F1-Score)
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Safe", "Jailbreak"]))